<a href="https://colab.research.google.com/github/bhar-gav/machine_learning/blob/main/Organizing_Hyperparameter_Sweeps_in_PyTorch_with_W%26B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Organizing_Hyperparameter_Sweeps_in_PyTorch_with_W&B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<!--- @wandbcode{sweeps-video} -->

In [1]:
!pip install wandb -Uq

2. Import W&B:

In [2]:
import wandb

3. Log in to W&B and provide your API key when prompted:

In [3]:
wandb.login()

<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bhar_gav (bhar_gav-national-institute-of-technology-hamirpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Pick a search method

First, specify a hyperparameter search method within your configuration dictionary. [There are three hyperparameter search strategies to choose from: grid, random, and Bayesian search](https://docs.wandb.ai/guides/sweeps/sweep-config-keys#method).

For this tutorial, you will use a random search. Within your notebook, create a dictionary and specify `random` for the `method` key.

In [4]:
sweep_config = {
    'method': 'random'
    }

Specify a metric that you want to optimize for. You do not need to specify the metric and goal for sweeps that use random search method. However, it is good practice to keep track of your sweep goals because you can refer to it at a later time.

In [5]:
metric = {
    'name': 'validation_accuracy',
    'goal': 'maximize'
    }

sweep_config['metric'] = metric

In [6]:
parameters_dict =  {
        'epochs': {'values': [ 5,10,15]},
        'hidden_layers': {'values': [3, 4, 5]},
        'hidden_size': {'values': [32, 64, 128]},
        'weight_decay': {'values': [0, 0.0005, 0.5]},
        'lr': {'values': [1e-3, 1e-4]},
        'optimizer': {'values': ['sgd', 'momentum', 'nesterov', 'rmsprop', 'adam', 'nadam', 'adadelta', 'adamw']},
        'batch_size': {'values': [16, 32, 64]},
        'activation': {'values': ['sigmoid', 'tanh', 'relu']},
        'weight_init': {'values': ['random', 'xavier']},
        'model': {'values': ['lenet']} #, 'resnet'
}

sweep_config['parameters'] = parameters_dict

In [7]:
import pprint
pprint.pprint(sweep_config)

{'method': 'random',
 'metric': {'goal': 'maximize', 'name': 'validation_accuracy'},
 'parameters': {'activation': {'values': ['sigmoid', 'tanh', 'relu']},
                'batch_size': {'values': [16, 32, 64]},
                'epochs': {'values': [5, 10, 15]},
                'hidden_layers': {'values': [3, 4, 5]},
                'hidden_size': {'values': [32, 64, 128]},
                'lr': {'values': [0.001, 0.0001]},
                'model': {'values': ['lenet']},
                'optimizer': {'values': ['sgd',
                                         'momentum',
                                         'nesterov',
                                         'rmsprop',
                                         'adam',
                                         'nadam',
                                         'adadelta',
                                         'adamw']},
                'weight_decay': {'values': [0, 0.0005, 0.5]},
                'weight_init': {'values': ['random',

## Step 2️: Initialize the Sweep

\

In [8]:
sweep_id = wandb.sweep(sweep_config, project="pytorch-sweeps")

Create sweep with ID: q6dn221t
Sweep URL: https://wandb.ai/bhar_gav-national-institute-of-technology-hamirpur/pytorch-sweeps/sweeps/q6dn221t


## Step 3:  Define your deep learning code



In [9]:
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
# Initialize device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset building function
def build_dataset(batch_size):
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])  # Normalize for MNIST
    dataset = datasets.MNIST('.', train=True, download=True, transform=transform)
    # Split 10% for validation
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


# Flexible LeNet model
class FlexibleLeNet(nn.Module):
    def __init__(self, hidden_layers, hidden_size, activation_function='relu', weight_init='random', output_size=10):
        super(FlexibleLeNet, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc_layers = []

        in_features = 16 * 4 * 4
        for _ in range(hidden_layers):
            self.fc_layers.append(nn.Linear(in_features, hidden_size))
            in_features = hidden_size
        self.fc_layers.append(nn.Linear(in_features, output_size))

        # Activation function
        self.activation_function = activation_function

        # Apply weight initialization
        self.apply(self._initialize_weights(weight_init))

    def _initialize_weights(self, init_type):
        def init_fn(m):
            if isinstance(m, nn.Linear):
                if init_type == 'xavier':
                    nn.init.xavier_uniform_(m.weight)
                else:
                    nn.init.normal_(m.weight, mean=0.0, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        return init_fn

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(x.size(0), -1)

        for fc in self.fc_layers:
            if self.activation_function == 'relu':
                x = F.relu(fc(x))
            elif self.activation_function == 'sigmoid':
                x = torch.sigmoid(fc(x))
            elif self.activation_function == 'tanh':
                x = torch.tanh(fc(x))
        return x





# Optimizer function
def get_optimizer(model, optimizer_name, lr, weight_decay=0):
    if optimizer_name == 'sgd':
        return optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'momentum':
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    elif optimizer_name == 'nesterov':
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=weight_decay)
    elif optimizer_name == 'rmsprop':
        return optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'adam':
        return optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'nadam':
        return optim.NAdam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'adadelta':
        return optim.Adadelta(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'adamw':
        return optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'swa':
        return optim.SWA(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError("Optimizer not supported")


# Training function
def train(model, train_loader, optimizer, criterion, epochs):
        config = wandb.config

        model.train()
        for epoch in range(epochs):
            running_loss = 0.0
            correct = 0
            total = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                # Forward pass
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

            wandb.log({
                "epoch": epoch + 1,
                "train_loss": running_loss / len(train_loader),
                "train_accuracy": 100 * correct / total,
                "trial_name": f"hl_{config.hidden_layers}_bs_{config.batch_size}_ac_{config.activation}"  # Add trial name
            })

            print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}, Accuracy: {100 * correct / total}%")

# Evaluation function
def evaluate(model, val_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy



In [10]:
def run_experiment():
    # Initialize a new wandb run
    with wandb.init(config=sweep_config)  as run:
        # If called by wandb.agent, as below,
        # this config will be set by Sweep Controller
        config = wandb.config


        # Generate a custom trial name using hyperparameters from the config
        trial_name = f"m_{config.model}_hl_{config.hidden_layers}_bs_{config.batch_size}_ac_{config.activation}_lr_{config.lr}_wd_{config.weight_decay}"

        run.name= trial_name

        # Initialize device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # Create dataset and loaders
        train_loader, val_loader = build_dataset(config.batch_size)

        # Choose model based on config
        if config.model == 'lenet':
            model = FlexibleLeNet(hidden_layers=config.hidden_layers, hidden_size=config.hidden_size, activation_function=config.activation, weight_init=config.weight_init).to(device)
        else:
            model = FlexibleResNet(hidden_layers=config.hidden_layers, hidden_size=config.hidden_size, activation_function=config.activation, weight_init=config.weight_init).to(device)

        # Define loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = get_optimizer(model, config.optimizer, config.lr, config.weight_decay)

        # Train the model
        train(model, train_loader, optimizer, criterion, config.epochs)

        # Evaluate the model
        val_accuracy = evaluate(model, val_loader)
        wandb.log({"validation_accuracy": val_accuracy, "trial_name": f"{trial_name}"})



# Run the sweep
wandb.agent(sweep_id, run_experiment,count=15)

wandb: Agent Starting Run: le5zc08l with config:
wandb: 	activation: relu
wandb: 	batch_size: 32
wandb: 	epochs: 5
wandb: 	hidden_layers: 3
wandb: 	hidden_size: 32
wandb: 	lr: 0.0001
wandb: 	model: lenet
wandb: 	optimizer: sgd
wandb: 	weight_decay: 0
wandb: 	weight_init: xavier
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:00<00:00, 16.6MB/s]


Extracting ./MNIST/raw/train-images-idx3-ubyte.gz to ./MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 527kB/s]


...

Epoch 1, Loss: 2.302406284481428, Accuracy: 9.798148148148147%
Epoch 2, Loss: 2.263509627484597, Accuracy: 13.012962962962963%
Epoch 3, Loss: 2.2105441104744283, Accuracy: 27.77037037037037%
Epoch 4, Loss: 2.1749871189560372, Accuracy: 35.07037037037037%
Epoch 5, Loss: 2.1535780421365494, Accuracy: 39.69259259259259%
Epoch 6, Loss: 2.13916476162689, Accuracy: 43.17037037037037%
Epoch 7, Loss: 2.1294411451895656, Accuracy: 45.54814814814815%
Epoch 8, Loss: 2.122194251742973, Accuracy: 47.36851851851852%
Epoch 9, Loss: 2.116289241901506, Accuracy: 48.69814814814815%
Epoch 10, Loss: 2.1112920076926174, Accuracy: 49.83148148148148%
Epoch 11, Loss: 2.106937647713304, Accuracy: 50.91111111111111%
Epoch 12, Loss: 2.1030654226434176, Accuracy: 51.79074074074074%
Epoch 13, Loss: 2.099554688727121, Accuracy: 52.596296296296295%
Epoch 14, Loss: 2.0963303017390285, Accuracy: 53.388888888888886%
Epoch 15, Loss: 2.093297584079453, Accuracy: 53.94814814814815%


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_accuracy,▁▂▄▅▆▆▇▇▇▇█████
train_loss,█▇▅▄▃▃▂▂▂▂▁▁▁▁▁
validation_accuracy,▁
epoch,15
train_accuracy,53.94815
train_loss,2.0933
trial_name,m_lenet_hl_3_bs_64_a...
validation_accuracy,55.66667


wandb: Agent Starting Run: 31justjb with config:
wandb: 	activation: tanh
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_layers: 3
wandb: 	hidden_size: 64
wandb: 	lr: 0.0001
wandb: 	model: lenet
wandb: 	optimizer: adadelta
wandb: 	weight_decay: 0.0005
wandb: 	weight_init: random


Epoch 1, Loss: 2.3069930989300764, Accuracy: 8.238888888888889%
Epoch 2, Loss: 2.3061128268065274, Accuracy: 8.777777777777779%
Epoch 3, Loss: 2.3052639403166593, Accuracy: 9.3%
Epoch 4, Loss: 2.304445442623562, Accuracy: 9.807407407407407%
Epoch 5, Loss: 2.3036511411313656, Accuracy: 10.277777777777779%
Epoch 6, Loss: 2.30287972718698, Accuracy: 10.75925925925926%
Epoch 7, Loss: 2.3021269016972297, Accuracy: 11.198148148148148%
Epoch 8, Loss: 2.3013881613413494, Accuracy: 11.666666666666666%
Epoch 9, Loss: 2.3006616443351464, Accuracy: 12.131481481481481%
Epoch 10, Loss: 2.299942192642777, Accuracy: 12.525925925925925%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▂▃▄▄▅▆▇▇█
train_loss,█▇▆▅▅▄▃▂▂▁
validation_accuracy,▁
epoch,10
train_accuracy,12.52593
train_loss,2.29994
trial_name,m_lenet_hl_3_bs_16_a...
validation_accuracy,11.9




---



end

## Visualize Sweep Results



### Parallel Coordinates Plot
This plot maps hyperparameter values to model metrics. It’s useful for honing in on combinations of hyperparameters that led to the best model performance.

![](https://assets.website-files.com/5ac6b7f2924c652fd013a891/5e190366778ad831455f9af2_s_194708415DEC35F74A7691FF6810D3B14703D1EFE1672ED29000BA98171242A5_1578695138341_image.png)


### Hyperparameter Importance Plot
The hyperparameter importance plot surfaces which hyperparameters were the best predictors of your metrics.
We report feature importance (from a random forest model) and correlation (implicitly a linear model).

![](https://assets.website-files.com/5ac6b7f2924c652fd013a891/5e190367778ad820b35f9af5_s_194708415DEC35F74A7691FF6810D3B14703D1EFE1672ED29000BA98171242A5_1578695757573_image.png)

These visualizations can help you save both time and resources running expensive hyperparameter optimizations by honing in on the parameters (and value ranges) that are the most important, and thereby worthy of further exploration.


## Learn more about W&B Sweeps

We created a simple training script and [a few flavors of sweep configs](https://github.com/wandb/examples/tree/master/examples/keras/keras-cnn-fashion) for you to play with. We highly encourage you to give these a try.

That repo also has examples to help you try more advanced sweep features like [Bayesian Hyperband](https://app.wandb.ai/wandb/examples-keras-cnn-fashion/sweeps/us0ifmrf?workspace=user-lavanyashukla), and [Hyperopt](https://app.wandb.ai/wandb/examples-keras-cnn-fashion/sweeps/xbs2wm5e?workspace=user-lavanyashukla).